# Clase 14A — Transformers, Embeddings y Semantic Search con dataset real
**Autor:** Josef Rodríguez  

Este notebook está diseñado como **masterclass completa**. Trabaja con el dataset real del curso:

`https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv`

## Objetivos de aprendizaje

Al finalizar esta clase, el alumno podrá:

1. Entender las diferencias entre representación clásica y representación moderna del texto.
2. Explicar qué es un embedding a nivel conceptual y matemático.
3. Aplicar embeddings sobre el dataset real de **planes de gobierno**.
4. Construir un sistema de **búsqueda semántica**.
5. Visualizar el espacio vectorial y analizar similitud entre candidatos y temas.

# 1. Motivación: del keyword search al semantic search

En NLP clásico, un sistema de búsqueda basado en palabras exactas funciona bien cuando la consulta y el documento comparten exactamente los mismos términos.

Ejemplo:

- Consulta: **educación**
- Documento: **mejorar la enseñanza escolar**

Aunque semánticamente están relacionados, un sistema literal puede fallar porque no entiende significado, solo coincidencias de tokens.

Por eso necesitamos una representación más rica del lenguaje.

## 1.1 Diagrama conceptual

```text
Texto
  ↓
Representación numérica
  ↓
Comparación matemática
  ↓
Recuperación de contenido similar
```

La calidad del sistema depende fuertemente de **cómo representamos el texto**.

# 2. Representación clásica del texto

## 2.1 Bag of Words (BoW)

Bag of Words representa un documento como un vector de frecuencias.

Si el vocabulario tiene tamaño \(V\), entonces cada documento se representa como:

\[
x_d \in \mathbb{{R}}^V
\]

donde cada posición indica cuántas veces aparece una palabra.

### Ventajas
- Fácil de implementar
- Interpretable
- Buen baseline

### Desventajas
- No captura orden
- No captura contexto
- No entiende sinónimos
- Produce vectores dispersos

## 2.2 TF–IDF

TF–IDF (Term Frequency – Inverse Document Frequency) es una técnica clásica de representación de texto que pondera las palabras según:

- su frecuencia dentro de un documento,
- y su rareza dentro del corpus completo.

La idea es que las palabras frecuentes en un documento pero raras en el corpus son más informativas.


### Fórmulas

#### Frecuencia de término (TF)

La frecuencia de término mide cuántas veces aparece un término \( t \) en un documento \( d \).

$$
TF(t,d) = \frac{f_{t,d}}{\sum_{t' \in d} f_{t',d}}
$$

donde:

- \( t \) = término  
- \( d \) = documento  
- \( f_{t,d} \) = número de veces que el término \( t \) aparece en el documento \( d \)  
- \( \sum_{t' \in d} f_{t',d} \) = número total de términos en el documento  


#### Frecuencia inversa de documento (IDF)

La frecuencia inversa de documento mide qué tan raro es un término en todo el corpus.

$$
IDF(t) = \log\left(\frac{N}{df(t)}\right)
$$

donde:

- \( N \) = número total de documentos en el corpus  
- \( df(t) \) = número de documentos que contienen el término \( t \)


#### Peso final TF–IDF

El peso final se obtiene multiplicando TF por IDF:

$$
TFIDF(t,d) = TF(t,d) \times IDF(t)
$$


### Intuición

- Si una palabra aparece muchas veces en un documento, **su TF aumenta**.
- Si una palabra aparece en muchos documentos del corpus, **su IDF disminuye**.
- Las palabras con alto **TF–IDF** suelen ser **las más informativas del documento**.

## 2.3 Limitaciones de TF-IDF

TF-IDF sigue siendo un enfoque superficial para tareas semánticas porque:

1. No modela el significado contextual.
2. Trata palabras similares como completamente distintas.
3. No entiende frases equivalentes.
4. Tiene dificultades para capturar relaciones semánticas profundas.

Por eso surge la idea de **embeddings densos**.

# 3. Embeddings: representación moderna del significado

Un **embedding** es una función que transforma texto en un vector numérico denso de dimensión fija.

Formalmente, podemos definir un embedding como una función:

$$
f : \text{Texto} \rightarrow \mathbb{R}^{d}
$$

donde:

- \( f \) es la función de embedding
- \( d \) es la dimensión del espacio vectorial
- $ \mathbb{R}^{d} $ denota el **espacio vectorial real de dimensión $d$**, es decir, el conjunto de todos los vectores  
  $ x = (x_1, x_2, ..., x_d) $ donde $ x_i \in \mathbb{R} $.

En la práctica, el valor de \( d \) suele ser mucho menor que el tamaño del vocabulario del corpus.

Ejemplos típicos de dimensiones:

- 300 (Word2Vec, GloVe)
- 384 o 768 (Sentence Transformers)
- 1024 o más (LLMs grandes)


## Idea clave

El principio fundamental de los embeddings es que **textos con significado similar deben quedar cerca en el espacio vectorial**.

Esto significa que la distancia entre sus vectores será pequeña según alguna métrica (por ejemplo, similitud coseno).


### Ejemplo intuitivo

```text
                ESPACIO SEMÁNTICO

        educación ●
           ● colegio
              ● aprendizaje


                                  ● minería
                              ● cobre
                                 ● exportación


        ● hospital
            ● salud
               ● medicina

## 3.1 Similaridad coseno

La métrica más utilizada para comparar embeddings es la **similitud coseno**, que mide el ángulo entre dos vectores en el espacio vectorial.

$$
\cos(\theta) = \frac{a \cdot b}{\|a\| \, \|b\|}
$$

donde:

- $a$ y $b$ son vectores en $\mathbb{R}^{d}$
- $a \cdot b$ es el **producto punto** entre los vectores
- $\|a\|$ y $\|b\|$ representan la **norma (magnitud)** de cada vector

La norma de un vector $a = (a_1, a_2, ..., a_d)$ se define como:

$$
\|a\| = \sqrt{\sum_{i=1}^{d} a_i^2}
$$


### Interpretación

- $\cos(\theta) \approx 1$ → vectores muy similares  
- $\cos(\theta) \approx 0$ → vectores aproximadamente ortogonales (sin relación semántica fuerte)  
- $\cos(\theta) < 0$ → vectores en direcciones opuestas

En sistemas de **semantic search**, se utiliza esta métrica para encontrar los documentos cuyos embeddings están más cerca del embedding de la consulta.

# 4. ¿Dónde entran los Transformers?

Los **Transformers** generan representaciones contextuales del lenguaje.  
A diferencia de modelos más antiguos, pueden capturar relaciones entre palabras incluso cuando están lejos dentro del texto.


## Pipeline simplificado

Texto  
↓  
Tokenización  
↓  
Embeddings de tokens  
↓  
Capas de self-attention  
↓  
Representación contextual  
↓  
Embedding de oración / documento  


## Idea de Self-Attention

El mecanismo central de los transformers es **self-attention**.

Para cada token de la secuencia, el modelo aprende **qué otros tokens del contexto son relevantes** para construir su representación.

De manera simplificada, el mecanismo de atención se define como:

$$
\text{Attention}(Q,K,V)=
\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

donde:

- $Q$ = matriz de **queries**
- $K$ = matriz de **keys**
- $V$ = matriz de **values**
- $d_k$ = dimensión de los vectores de key

El término $\sqrt{d_k}$ se utiliza para **normalizar el producto punto** y evitar valores demasiado grandes que dificulten el entrenamiento.


## Intuición

El mecanismo de atención permite que cada palabra determine **qué otras palabras del contexto son importantes**.

Ejemplo:

The cat sat on the mat

Cuando el modelo procesa **sat**, puede asignar mayor peso a **cat**, porque es el sujeto de la oración.


## Multi-Head Attention

En la práctica, los transformers usan **múltiples mecanismos de atención en paralelo** para capturar distintos tipos de relaciones semánticas.

$$
\text{MultiHead}(Q,K,V)=
\text{Concat}(head_1,\ldots,head_h)W^O
$$

donde cada cabeza de atención aprende un patrón distinto de relación entre palabras.


## Conexión con embeddings

Después de varias capas de self-attention, el modelo produce **representaciones contextuales del texto**.

Estas representaciones pueden agregarse para formar un **embedding de oración o documento**.

Estos embeddings se utilizan en tareas como:

- búsqueda semántica  
- clustering de documentos  
- sistemas RAG  
- recomendadores de contenido  
- recuperación de información

# 5. Trabajaremos con el dataset real del curso

El dataset contiene una columna con el nombre del candidato presidencial y otra con el texto de su plan de gobierno.

Nuestro objetivo será responder preguntas como:

- ¿Qué candidatos hablan más de educación?
- ¿Qué candidatos mencionan salud o hospitales?
- ¿Qué planes se parecen más entre sí?
- ¿Qué fragmentos son más relevantes para una consulta específica?

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
df.columns.tolist(), df.shape

# 6. Exploración inicial del dataset

En una masterclass no basta con cargar el dataset: debemos entenderlo.

Analizaremos:

1. número de documentos
2. longitud de texto por candidato
3. presencia de temas frecuentes
4. calidad general del texto

In [ ]:
df = df.rename(columns={df.columns[0]: "presidente", df.columns[1]: "texto_plan"})
df["texto_plan"] = df["texto_plan"].fillna("").astype(str)
df["n_chars"] = df["texto_plan"].str.len()
df["n_words"] = df["texto_plan"].str.split().str.len()

df[["presidente", "n_chars", "n_words"]].head()

In [ ]:
df[["n_chars", "n_words"]].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df["n_words"], bins=20)
plt.title("Distribución de número de palabras por plan")
plt.xlabel("Número de palabras")
plt.ylabel("Frecuencia")
plt.show()

# 7. Baseline clásico con TF-IDF

Antes de usar embeddings, construimos un baseline sólido.

Esto es importante porque en una clase maestra no solo mostramos la tecnología nueva; también enseñamos **contra qué se compara**.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words=None,
    max_features=5000,
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(df["texto_plan"])
X_tfidf.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "educación pública y calidad educativa"
q_vec = tfidf.transform([query])
scores_tfidf = cosine_similarity(q_vec, X_tfidf).flatten()

top_idx = np.argsort(scores_tfidf)[::-1][:5]
df.loc[top_idx, ["presidente"]].assign(score=scores_tfidf[top_idx])

## Reflexión pedagógica

TF-IDF puede recuperar resultados razonables, pero sigue dependiendo fuertemente de coincidencias léxicas.  
Ahora construiremos una versión semántica con embeddings.

# 8. Sentence Embeddings sobre el dataset real

Usaremos `sentence-transformers`, que permite obtener embeddings de texto listos para tareas de similitud semántica.

## ¿Qué modelo usaremos?

`all-MiniLM-L6-v2` es una opción muy útil para docencia porque:
- es relativamente liviano,
- funciona bien en similitud semántica,
- permite experimentar rápido.

In [ ]:
# Si estás en Colab o un entorno limpio, descomenta:
# !pip install -q sentence-transformers umap-learn

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(
    df["texto_plan"].tolist(),
    show_progress_bar=True
)

embeddings.shape

# 9. Búsqueda semántica con embeddings

El pipeline será:

```text
consulta del usuario
        ↓
embedding de la consulta
        ↓
cosine similarity contra embeddings de los planes
        ↓
ranking de candidatos
```

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def buscar_semanticamente(query, top_k=5):
    q_emb = model.encode([query])
    scores = cosine_similarity(q_emb, embeddings).flatten()
    idx = np.argsort(scores)[::-1][:top_k]
    return df.loc[idx, ["presidente"]].assign(score=scores[idx])

buscar_semanticamente("educación pública y aprendizaje", top_k=5)

In [ ]:
buscar_semanticamente("hospitales, salud pública y atención primaria", top_k=5)

In [ ]:
buscar_semanticamente("minería, exportaciones y crecimiento económico", top_k=5)

# 10. Visualización del espacio semántico

Los embeddings viven en alta dimensión, pero podemos proyectarlos a 2D para tener intuición visual.

Usaremos:
- PCA como reducción lineal simple
- UMAP como técnica no lineal más potente

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_2d_pca = pca.fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(emb_2d_pca[:,0], emb_2d_pca[:,1])
plt.title("Proyección PCA de planes de gobierno")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
# Si tienes instalado umap-learn:
# import umap
# reducer = umap.UMAP(random_state=42)
# emb_2d_umap = reducer.fit_transform(embeddings)
# plt.figure(figsize=(8,6))
# plt.scatter(emb_2d_umap[:,0], emb_2d_umap[:,1])
# plt.title("Proyección UMAP de planes de gobierno")
# plt.show()

# 11. Análisis de similitud entre candidatos

Otra pregunta interesante de negocio o ciencia política es:

> ¿Qué planes de gobierno son más parecidos entre sí?

Esto se puede calcular comparando los embeddings de los documentos completos.

In [ ]:
sim_matrix = cosine_similarity(embeddings)
sim_matrix.shape

In [ ]:
sim_df = pd.DataFrame(sim_matrix, index=df["presidente"], columns=df["presidente"])
sim_df.iloc[:5, :5]

In [ ]:
# Ejemplo: top candidatos más parecidos al primero
candidato_base = df["presidente"].iloc[0]
similares = sim_df.loc[candidato_base].sort_values(ascending=False).head(6)
similares

# 12. Clustering semántico

Si los embeddings capturan significado, entonces podemos agrupar planes parecidos.

Esto conecta embeddings con aprendizaje no supervisado.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embeddings)

df["cluster_embedding"] = clusters
df[["presidente", "cluster_embedding"]].head(10)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(emb_2d_pca[:,0], emb_2d_pca[:,1], c=df["cluster_embedding"])
plt.title("Clusters semánticos sobre embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# 13. Comparación conceptual: TF-IDF vs Embeddings

## TF-IDF
- representación dispersa
- basada en frecuencia
- interpretable
- útil como baseline

## Embeddings
- representación densa
- captura semántica
- funciona mejor para búsqueda semántica
- base de RAG y sistemas modernos de IA

## Mensaje de masterclass
No se trata de reemplazar ciegamente TF-IDF por embeddings.  
Se trata de entender **cuándo** un enfoque clásico basta y **cuándo** un enfoque semántico agrega valor real.

# 14. Conclusiones de la clase

En esta clase aprendimos que:

1. El texto necesita convertirse a números para ser procesado.
2. TF-IDF fue un gran avance, pero tiene límites semánticos.
3. Los embeddings permiten representar significado.
4. Con embeddings podemos construir sistemas de semantic search.
5. El dataset real de planes de gobierno es ideal para este tipo de análisis.

## Frase de cierre
**Un embedding no es magia: es una representación matemática del significado.**